In [7]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [8]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [9]:
# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "stackoverflow" dataset
dataset_ref = client.dataset("stackoverflow", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "posts_questions" table
table_ref = dataset_ref.table("posts_questions")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(table, max_results=5).to_dataframe()

,id,title,body,accepted_answer_id,answer_count,comment_count,community_owned_date,creation_date,favorite_count,last_activity_date,last_edit_date,last_editor_display_name,last_editor_user_id,owner_display_name,owner_user_id,parent_id,post_type_id,score,tags,view_count
0,320268,Html.ActionLink doesn’t render # properly,<p>When using Html.ActionLink passing a string...,<NA>,0,0,NaT,2008-11-26 10:42:37.477000+00:00,0,2009-02-06 20:13:54.370000+00:00,NaT,NaN,<NA>,Paulo,<NA>,NaN,1,0,asp.net-mvc,390
1,324003,Primitive recursion,<p>how will i define the function 'simplify' ...,<NA>,0,0,NaT,2008-11-27 15:12:37.497000+00:00,0,2012-09-25 19:54:40.597000+00:00,2012-09-25 19:54:40.597000+00:00,Marcin,1288,NaN,41000,NaN,1,0,haskell|lambda|functional-programming|lambda-c...,497
2,390605,While vs. Do While,<p>I've seen both the blocks of code in use se...,390608,0,0,NaT,2008-12-24 01:49:54.230000+00:00,2,2008-12-24 03:08:55.897000+00:00,NaT,NaN,<NA>,Unkwntech,115,NaN,1,0,language-agnostic|loops,11262
3,413246,Protect ASP.NET Source code,<p>Im currently doing some research in how to ...,<NA>,0,0,NaT,2009-01-05 14:23:51.040000+00:00,0,2009-03-24 21:30:22.370000+00:00,2009-01-05 14:42:28.257000+00:00,Tom Anderson,13502,Velnias,<NA>,NaN,1,0,asp.net|deployment|obfuscation,4823
4,454921,"Difference between ""int[] myArray"" and ""int my...",<blockquote>\n <p><strong>Possible Duplicate:...,454928,0,0,NaT,2009-01-18 10:22:52.177000+00:00,0,2009-01-18 10:30:50.930000+00:00,2017-05-23 11:49:26.567000+00:00,NaN,-1,Evan Fosmark,49701,NaN,1,0,java|arrays,798


In [10]:
# SQL Query to calculate how fast questions get answered.
# We define first_query as a multi-line string containing our SQL commands.
first_query = """
              # SELECT the question ID
              SELECT q.id AS q_id,
                  # Calculate the time difference (in seconds) between when the question was asked
                  # and when its answers were created. We use MIN() to get the time of the VERY FIRST answer.
                  MIN(TIMESTAMP_DIFF(a.creation_date, q.creation_date, SECOND)) as time_to_answer
              # From the 'posts_questions' table (aliased as 'q')
              FROM `bigquery-public-data.stackoverflow.posts_questions` AS q
                  # INNER JOIN matches questions with their answers in the 'posts_answers' table (aliased as 'a')
                  INNER JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
              # The connection point: the answer's parent_id must match the question's id
              ON q.id = a.parent_id
              # Only look at questions asked between January 1, 2018 and January 31, 2018
              WHERE q.creation_date >= '2018-01-01' and q.creation_date < '2018-02-01'
              # Group the results by question ID so we find the MIN time per unique question
              GROUP BY q_id
              # Sort the results by the fastest answer time
              ORDER BY time_to_answer
              """

# Run the query using our client, and convert the results into a Pandas DataFrame for easy analysis.
# create_bqstorage_client=False bypasses the BigQuery Storage API which requires special permissions.
first_result = client.query(first_query).result().to_dataframe(create_bqstorage_client=False)

# Calculate the percentage of questions that actually received an answer.
# 1. sum(first_result["time_to_answer"].notnull()) counts how many questions have a valid answer time.
# 2. len(first_result) counts the total number of questions returned by the query.
# 3. Divide them and multiply by 100 to get a percentage.
print("Percentage of answered questions: %s%%" % \
      (sum(first_result["time_to_answer"].notnull()) / len(first_result) * 100))

# Print the total number of questions retrieved.
print("Number of questions:", len(first_result))

# Display the first 5 rows of our resulting DataFrame to preview the data.
first_result.head()


Percentage of answered questions: 100.0%
Number of questions: 134719


,q_id,time_to_answer
0,48382183,-132444692
1,48375126,0
2,48174391,0
3,48092100,0
4,48102324,0


In [11]:
# The INNER JOIN above silently dropped every question that had no answer,
# which is why we (wrongly) saw 100% answered and far too few questions.
# Switching to a LEFT JOIN keeps ALL questions from posts_questions (the left table),
# regardless of whether a matching answer exists. Unanswered questions get a NULL
# time_to_answer, so the answered-percentage falls below 100% and the row count climbs.
correct_query = """ 
                # SELECT the question ID
                SELECT q.id AS q_id,
                    # Time (in seconds) from the question to its VERY FIRST answer.
                    # NULL for questions that were never answered.
                    MIN(TIMESTAMP_DIFF(a.creation_date, q.creation_date, SECOND)) as time_to_answer
                # From the 'posts_questions' table (aliased as 'q')
                FROM `bigquery-public-data.stackoverflow.posts_questions` AS q
                    # LEFT JOIN keeps every question even when it has no matching answer
                    LEFT JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
                # The connection point: the answer's parent_id must match the question's id
                ON q.id = a.parent_id
                # Only look at questions asked between January 1, 2018 and January 31, 2018
                WHERE q.creation_date >= '2018-01-01' and q.creation_date < '2018-02-01'
                # Group the results by question ID so we find the MIN time per unique question
                GROUP BY q_id
                # Sort the results by the fastest answer time
                ORDER BY time_to_answer
                """

# Run the query, and return a pandas DataFrame.
correct_result = client.query(correct_query).result().to_dataframe(create_bqstorage_client=False)
print("Percentage of answered questions: %s%%" % \
      (sum(correct_result["time_to_answer"].notnull()) / len(correct_result) * 100))
print("Number of questions:", len(correct_result))   


Percentage of answered questions: 83.3368387192557%
Number of questions: 161656


In [ ]:
three_tables_query = """
                     -- 1. What columns do we want in our final table?
                     -- We want the user ID, and the EARLIEST (MIN) question and answer dates.
                     SELECT 
                         u.id AS id,
                         MIN(q.creation_date) AS q_creation_date,
                         MIN(a.creation_date) AS a_creation_date
                         
                     -- 2. What is our "Base" table? 
                     -- We start with the users table because it has our master list of accounts.
                     FROM `bigquery-public-data.stackoverflow.users` AS u
                     
                     -- 3. Attach the Questions table
                     -- We use a LEFT JOIN so we don't lose users who never asked a question.
                     LEFT JOIN `bigquery-public-data.stackoverflow.posts_questions` AS q
                         ON u.id = q.owner_user_id
                         
                     -- 4. Attach the Answers table
                     -- We use another LEFT JOIN so we don't lose users who never gave an answer.
                     LEFT JOIN `bigquery-public-data.stackoverflow.posts_answers` AS a
                         ON u.id = a.owner_user_id
                         
                     -- 5. Filter our Base list
                     -- We ONLY want to look at users who created their account in January 2019.
                     WHERE u.creation_date >= '2019-01-01' AND u.creation_date < '2019-02-01'
                     
                     -- 6. Group the results
                     -- Because we are calculating a MIN() date per user, we have to tell 
                     -- SQL to group all the raw data by the user's ID before finding the minimum.
                     GROUP BY id
                     """

# Run the query, and convert the results to a pandas DataFrame.
# Note: Remember to keep create_bqstorage_client=False to bypass the Storage API permission error
three_tables_result = client.query(three_tables_query).result().to_dataframe(create_bqstorage_client=False)

# Print the total number of users retrieved.
print("Number of users:", len(three_tables_result))

# Display the first 5 rows.
three_tables_result.head()


In [ ]:
# Query to find all distinct users who posted on January 1, 2019.
all_users_query = """
                  -- Get user IDs of everyone who ASKED a question on Jan 1, 2019
                  SELECT owner_user_id
                  FROM `bigquery-public-data.stackoverflow.posts_questions`
                  WHERE EXTRACT(DATE FROM creation_date) = '2019-01-01'

                  -- UNION DISTINCT combines both result sets and automatically
                  -- removes any duplicate user IDs
                  UNION DISTINCT

                  -- Get user IDs of everyone who ANSWERED a question on Jan 1, 2019
                  SELECT owner_user_id
                  FROM `bigquery-public-data.stackoverflow.posts_answers`
                  WHERE EXTRACT(DATE FROM creation_date) = '2019-01-01'
                  """